# Manila Rainfall Prediction – Full Pipeline
**Course:** Water Conservation and Rainwater Harvesting (BCV605A)

Runs the complete end-to-end pipeline:
1. Setup
2. Preprocessing
3. Feature Engineering
4. Model Training (Train 1995-2020 80:20 | Unseen 2021-2025)
5. Evaluation & All Charts
6. Unseen Prediction (2021-2025)
7. Inspect Pickle File

## 1. Setup

In [ ]:
import sys
import os

# Make sure src/ is importable
sys.path.append(os.path.abspath('../'))
sys.path.append(os.path.abspath('../src'))

from src.preprocessing import preprocess_data
from src.feature_engineering import engineer_features
from src.train import train_models
from src.evaluate import generate_evaluation_plots
from src.predict import predict_unseen
import joblib
import pandas as pd
import numpy as np

# Project root
BASE_DIR = os.path.abspath('../')
print('Project root:', BASE_DIR)

## 2. Preprocessing

In [ ]:
raw_file = os.path.join(BASE_DIR, 'data', 'raw', 'Manila.csv')
if os.path.exists(raw_file):
    df_cleaned, scaler = preprocess_data(raw_file)
    models_dir = os.path.join(BASE_DIR, 'models')
    os.makedirs(models_dir, exist_ok=True)
    joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
    # Save cleaned data
    processed_dir = os.path.join(BASE_DIR, 'data', 'processed')
    os.makedirs(processed_dir, exist_ok=True)
    df_cleaned.to_csv(os.path.join(processed_dir, 'cleaned_data.csv'), index=False)
    print('Preprocessing complete. Shape:', df_cleaned.shape)
else:
    print('Raw data not found! Run: python src/fetch_data.py')

## 3. Feature Engineering

In [ ]:
df_feat = engineer_features(df_cleaned)
processed_dir = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
df_feat.to_csv(os.path.join(processed_dir, 'feature_engineered_data.csv'), index=False)
print('Feature engineering complete. Shape:', df_feat.shape)
print('Columns:', df_feat.columns.tolist())

## 4. Model Training
- **Train/Test**: 1995–2020, 80:20 year-wise random split
- **Unseen**: 2021–2025 (never used in training)

In [ ]:
results_df, best_model_name, models, pca, \
    train_data, test_data, unseen_data, feature_cols = train_models(df_feat)

print(f'\nBest model: {best_model_name}')
print('\nFull comparison table:')
display(results_df)

## 5. Evaluation & All Charts

In [ ]:
best_model = joblib.load(os.path.join(BASE_DIR, 'models', 'best_model.pkl'))
pca_loaded = joblib.load(os.path.join(BASE_DIR, 'models', 'pca.pkl'))

generate_evaluation_plots(df_feat, best_model, best_model_name, pca_loaded)
print('All evaluation plots generated in outputs/figures/')

## 6. Unseen Prediction (2021–2025)

In [ ]:
scaler_loaded = joblib.load(os.path.join(BASE_DIR, 'models', 'scaler.pkl'))
pred_df = predict_unseen(df_feat, best_model, scaler_loaded, pca_loaded, best_model_name)

print('\nSample predictions (first 10 rows):')
display(pred_df.head(10))

## 7. Inspect Pickle File (Report Section 10)

In [ ]:
import joblib

model_path  = os.path.join(BASE_DIR, 'models', 'best_model.pkl')
scaler_path = os.path.join(BASE_DIR, 'models', 'scaler.pkl')

m = joblib.load(model_path)
s = joblib.load(scaler_path)

print('=== Best Model ===')
print('Type :', type(m))
print(m)

print('\n=== Scaler ===')
print('Type :', type(s))
print('Features fitted on:', s.feature_names_in_.tolist())